# Participant starter: build your ISU system (Track B)

This notebook provides the participant interface for Track B. It loads the starter-kit images and labels, calls your `isu_system()` wrapper for each image, and saves predictions as JSON.

Your job is to replace the marked wrapper body with your own in-car scene understanding system. The reference implementation uses `Qwen/Qwen2.5-VL-3B-Instruct` so the notebook runs end to end as a baseline.

See the root `README.md` and `track_b_evaluator.ipynb` for the feature definitions and evaluation metrics.

## 1. Configure the environment

Install the Track B dependencies before running the notebook:

```bash
pip install -r track_b/requirements.txt
```

In [2]:
import json
import sys
import time
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, Qwen2_5_VLForConditionalGeneration

NOTEBOOK_DIR = Path.cwd()
TRACK_B_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'track_b_evaluator.py').exists() else NOTEBOOK_DIR / 'track_b'
REPO_ROOT = TRACK_B_DIR.parent
DATA_DIR = REPO_ROOT / 'data'
DATASET_ID = 'ISU-Test/isu-challenge-dataset'

if str(TRACK_B_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_B_DIR))

from track_b_evaluator import (
    DEFAULT_IGNORED_FEATURES,
    build_prompt,
    build_vocabulary,
    comparable_features,
    load_dataset,
    parse_json_answer,
    query_model,
)

# Download the dataset outside track_b when it is not already available locally.
if not any(DATA_DIR.rglob('*_sim.png')):
    print(f'Downloading {DATASET_ID} to {DATA_DIR}...')
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type='dataset',
        local_dir=str(DATA_DIR),
    )

# Choose 'qwen' for the stronger reference model or 'moondream' for a lighter option.
MODEL_BACKEND = 'qwen'
MODEL_NAME = (
    'Qwen/Qwen2.5-VL-3B-Instruct'
    if MODEL_BACKEND == 'qwen'
    else 'vikhyatk/moondream2'
)
IMAGE_KIND = 'both'
SCENE_LIMIT = None
OUTPUT_PATH = TRACK_B_DIR / 'submissions' / 'track_b_predictions.json'
IGNORED_FEATURES = DEFAULT_IGNORED_FEATURES.copy()
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Starter kit data directory: {DATA_DIR}')
print(f'Model: {MODEL_NAME}')
print(f'Device: {DEVICE}')

ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub' (C:\Users\Lev Sorokin\Documents\testing\ISU-Challenge\Starterkit-ICSE-2027\track_b\venv\Lib\site-packages\huggingface_hub\__init__.py)

## 2. Load the labeled data

The notebook evaluates every discovered image-label pair. Set `SCENE_LIMIT` above to a small number for a smoke test. Leave it as `None` to process all scenes.

In [ ]:
scenes = load_dataset(DATA_DIR, IMAGE_KIND)
if SCENE_LIMIT is not None:
    scenes = scenes[:SCENE_LIMIT]
vocabulary = build_vocabulary(scenes, IGNORED_FEATURES)
PROMPT = build_prompt(vocabulary)

print(f'Loaded {len(scenes)} image-label pairs')
print('Features:', ', '.join(vocabulary))

## 3. Choose a model and implement the SUT wrapper

Set `MODEL_BACKEND` in the setup cell to `qwen` for the reference Qwen2.5-VL-3B-Instruct model or `moondream` for the lighter `vikhyatk/moondream2` model. Both options use the same `isu_system(image)` interface.

Implement `isu_system(image)` below. It receives one RGB `PIL.Image.Image` and must return a JSON-compatible dictionary with the requested feature names and one classification value for each feature.

The marked body is the participant implementation point. The reference body supports both example models and can be replaced by any reproducible model or pipeline.

In [ ]:
if MODEL_BACKEND == 'qwen':
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype='auto',
        device_map=DEVICE,
    ).eval()
else:
    processor = None
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision='2025-06-21',
        trust_remote_code=True,
    ).to(DEVICE).eval()


def isu_system(image: Image.Image) -> dict:
    """Return one feature classification for one RGB image."""
    # TODO: Participants must implement this wrapper with their own ISU system.
    # This reference implementation supports Qwen and the lighter Moondream option.
    if MODEL_BACKEND == 'qwen':
        raw_answer = query_model(model, processor, image, PROMPT, DEVICE)
    else:
        raw_answer = model.query(
            image,
            PROMPT,
            reasoning=False,
            settings={'temperature': 0.0, 'top_p': 0.0, 'max_tokens': 768},
        )['answer']
    prediction = parse_json_answer(raw_answer)
    if prediction is None:
        raise ValueError('isu_system() must return a valid JSON object.')
    return prediction

## 4. Generate predictions

The evaluator calls the same `isu_system(image)` interface. Each output records the image name, prediction, expected labels, feature matches, and inference time.

In [ ]:
prediction_rows = []

for index, (scene_name, image_path, label) in enumerate(scenes, 1):
    image = Image.open(image_path).convert('RGB')
    started = time.perf_counter()
    prediction = isu_system(image)
    expected = comparable_features(label, IGNORED_FEATURES)
    feature_matches = {
        key: str(prediction.get(key)).strip().upper() == str(value).strip().upper()
        for key, value in expected.items()
    }
    prediction_rows.append({
        'image': scene_name,
        'image_path': str(image_path),
        'expected': expected,
        'prediction': prediction,
        'feature_matches': feature_matches,
        'all_features_match': all(feature_matches.values()),
        'inference_seconds': time.perf_counter() - started,
    })
    print(f'[{index}/{len(scenes)}] {scene_name}', flush=True)

## 5. Save predictions

The resulting JSON can be inspected locally and supplied to the Track B evaluation workflow.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
report = {
    'track': 'B',
    'model': MODEL_NAME,
    'device': DEVICE,
    'features': vocabulary,
    'predictions': prediction_rows,
}
OUTPUT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=True))
print(f'Saved predictions to {OUTPUT_PATH}')